
# Skadi BI - Bronze

Fase inicial: lê os dados necessários do PostgreSQL operacional e cria as tabelas Bronze.
As credenciais são lidas de Databricks Secrets, nunca registradas no Git.

In [ ]:
from pyspark.sql import DataFrame

spark.conf.set("spark.sql.session.timeZone", "America/Sao_Paulo")

dbutils.widgets.text("secret_scope", "")
dbutils.widgets.text("secret_jdbc_url", "")
dbutils.widgets.text("secret_usuario", "")
dbutils.widgets.text("secret_senha", "")
dbutils.widgets.text("schema_origem", "public")

SECRET_SCOPE = dbutils.widgets.get("secret_scope").strip()
SECRET_JDBC_URL = dbutils.widgets.get("secret_jdbc_url").strip()
SECRET_USUARIO = dbutils.widgets.get("secret_usuario").strip()
SECRET_SENHA = dbutils.widgets.get("secret_senha").strip()
SCHEMA_ORIGEM = dbutils.widgets.get("schema_origem").strip()

if not all([SECRET_SCOPE, SECRET_JDBC_URL, SECRET_USUARIO, SECRET_SENHA, SCHEMA_ORIGEM]):
    raise ValueError("Preencha os widgets de conexão antes de executar o notebook Bronze.")

JDBC_URL = dbutils.secrets.get(SECRET_SCOPE, SECRET_JDBC_URL)
JDBC_USUARIO = dbutils.secrets.get(SECRET_SCOPE, SECRET_USUARIO)
JDBC_SENHA = dbutils.secrets.get(SECRET_SCOPE, SECRET_SENHA)


def ler_postgres(consulta: str) -> DataFrame:
    return (
        spark.read.format("jdbc")
        .option("url", JDBC_URL)
        .option("dbtable", f"({consulta}) AS origem")
        .option("user", JDBC_USUARIO)
        .option("password", JDBC_SENHA)
        .option("driver", "org.postgresql.Driver")
        .option("fetchsize", "10000")
        .load()
    )


def gravar_bronze(df: DataFrame, nome_tabela: str) -> None:
    (
        df.write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(nome_tabela)
    )

In [ ]:
usuarios = ler_postgres(
    f"SELECT id, nivel_acesso FROM {SCHEMA_ORIGEM}.tb_usuario"
)

chamados = ler_postgres(
    f"""
    SELECT id, cod_usuario_solicitante, categoria, status,
           data_hora_abertura, data_hora_encerramento,
           cod_usuario_responsavel
    FROM {SCHEMA_ORIGEM}.tb_solicitacao_suporte
    """
)

controle_rpa = ler_postgres(
    f"SELECT nome_carga, ultima_execucao_sucesso "
    f"FROM {SCHEMA_ORIGEM}.tb_controle_rpa"
)

gravar_bronze(usuarios, "bronze_usuario")
gravar_bronze(chamados, "bronze_solicitacao_suporte")
gravar_bronze(controle_rpa, "bronze_controle_rpa")

In [ ]:
display(spark.table("bronze_usuario"))
display(spark.table("bronze_solicitacao_suporte"))
display(spark.table("bronze_controle_rpa"))